# 06 — Risk Attribution: Euler Decomposition & Risk Budgeting

**Risk attribution** decomposes total portfolio risk into per-asset contributions, answering the question: *"Which assets contribute the most to portfolio VaR?"*

### Euler Decomposition

Portfolio VaR (parametric) is a **homogeneous function of degree 1** in the weights: scaling all weights by $\lambda$ scales VaR by $\lambda$. By Euler's theorem:

$$
\text{VaR}(\mathbf{w}) = \sum_{i=1}^{n} w_i \cdot \frac{\partial \text{VaR}}{\partial w_i} = \sum_{i=1}^{n} \text{CVaR}_i
$$

where:
- **Marginal VaR** (M-VaR): $\frac{\partial \text{VaR}}{\partial w_i} = z_\alpha \frac{(\Sigma \mathbf{w})_i}{\sqrt{\mathbf{w}^T \Sigma \mathbf{w}}}$ — the rate of change of portfolio VaR per unit weight change.
- **Component VaR** (C-VaR): $w_i \times \text{M-VaR}_i$ — the additive contribution of asset $i$ to total VaR.

### Risk Budgeting

An **Equal Risk Contribution (ERC)** portfolio targets $\text{CVaR}_i = \text{VaR} / n$ for all assets. In practice, portfolios deviate significantly from ERC — a few high-volatility or high-correlation assets often dominate risk despite modest weights.

---

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from var_risk_engine.data import fetch_and_prepare
from var_risk_engine.covariance import sample_covariance
from var_risk_engine.risk_attribution import (
    risk_attribution_summary,
    risk_attribution_table,
    risk_budget_analysis,
    marginal_var,
    component_var,
    risk_contribution_pct,
    incremental_var,
)
from var_risk_engine.var_parametric import parametric_var_from_cov

%matplotlib inline

# Color palette
PRIMARY    = "#1B3A5C"
SECONDARY  = "#E8734A"
TERTIARY   = "#4CAF50"
QUATERNARY = "#9C27B0"
PALETTE    = [PRIMARY, SECONDARY, TERTIARY, QUATERNARY, "#FFC107"]

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# --- Define portfolio: 5 assets with weights ---
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
weights = np.array([0.30, 0.25, 0.20, 0.15, 0.10])
confidence = 0.95

# Fetch data
prices, returns = fetch_and_prepare(tickers, start="2020-01-01")

print("Portfolio Specification")
print("=" * 50)
for t, w in zip(tickers, weights):
    print(f"  {t:6s}  weight = {w:.0%}")
print(f"\nPeriod:        {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Observations:  {len(returns)}")
print(f"Confidence:    {confidence:.0%}")

# Covariance matrix
cov_matrix = sample_covariance(returns)
port_sigma = np.sqrt(weights @ cov_matrix @ weights)
port_var = parametric_var_from_cov(weights, cov_matrix, confidence)

print(f"\nPortfolio volatility:  {port_sigma:.6f}")
print(f"Portfolio VaR ({confidence:.0%}):   {port_var:.6f}")

In [ ]:
# --- Full risk attribution summary ---
summary = risk_attribution_summary(tickers, weights, returns, confidence=confidence)

print("=" * 70)
print("  Risk Attribution Summary")
print("=" * 70)
print(f"  Portfolio VaR ({confidence:.0%}):  {summary['portfolio_var']:.6f}")
print(f"  Concentration (HHI):   {summary['concentration']:.4f}")
print(f"  (HHI = 1/n = {1/len(tickers):.4f} for perfect diversification)")
print(f"  (HHI = 1.0 for complete concentration)")
print("=" * 70)

# Display the attribution table
attr_table = summary["attribution_table"]
print("\nRisk Attribution Table (sorted by risk contribution):")
print("-" * 100)
display_cols = ["ticker", "weight_pct", "component_var",
                "marginal_var", "risk_contribution_pct"]
print(attr_table[display_cols].to_string(index=False, float_format="%.4f"))
print("-" * 100)

In [ ]:
# --- Risk budget analysis: current vs equal-risk target ---
risk_budget = summary["risk_budget"]

print("\nRisk Budget Analysis: Current vs Equal Risk Contribution (ERC)")
print("=" * 70)
print(risk_budget.to_string(index=False, float_format="%.2f"))
print("=" * 70)

# Highlight the largest deviations
max_over = risk_budget.loc[risk_budget["deviation_pct"].idxmax()]
max_under = risk_budget.loc[risk_budget["deviation_pct"].idxmin()]

print(f"\n  Largest over-contribution:  {max_over['ticker']} "
      f"({max_over['deviation_pct']:+.2f} pp)")
print(f"  Largest under-contribution: {max_under['ticker']} "
      f"({max_under['deviation_pct']:+.2f} pp)")

print(f"\n  An Equal Risk Contribution portfolio would require rebalancing")
print(f"  toward lower-vol assets and away from high-risk-contribution assets.")

In [ ]:
# --- Incremental VaR per asset ---
ivar = summary["incremental_var"]

ivar_df = pd.DataFrame({
    "Asset": tickers,
    "Weight": weights,
    "Incremental_VaR": ivar,
    "I_VaR_pct_of_total": ivar / summary["portfolio_var"] * 100,
}).sort_values("Incremental_VaR", ascending=False).reset_index(drop=True)

print("\nIncremental VaR: Portfolio VaR reduction if asset is removed")
print("(removed weight is redistributed equally among remaining assets)")
print("=" * 70)
print(ivar_df.to_string(index=False, float_format="%.5f"))
print("=" * 70)

print(f"\n  Note: Incremental VaR is NOT additive.")
print(f"  Sum of I-VaR = {ivar.sum():.6f} vs Portfolio VaR = {summary['portfolio_var']:.6f}")
print(f"  This is because removing one asset changes the diversification")
print(f"  benefit for all remaining assets.")

# Marginal VaR comparison
mvar = marginal_var(weights, cov_matrix, confidence)
print(f"\n  Marginal VaR (rate of change of VaR per unit weight):")
for t, mv in sorted(zip(tickers, mvar), key=lambda x: -x[1]):
    print(f"    {t:6s}  M-VaR = {mv:.5f}")

In [ ]:
# --- Pie chart: risk contributions ---
rpct = risk_contribution_pct(weights, cov_matrix, confidence)

# Sort by contribution for the chart
sorted_idx = np.argsort(rpct)[::-1]
sorted_tickers = [tickers[i] for i in sorted_idx]
sorted_rpct = rpct[sorted_idx]
sorted_colors = [PALETTE[tickers.index(t)] for t in sorted_tickers]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Risk contribution pie
wedges, texts, autotexts = axes[0].pie(
    sorted_rpct,
    labels=sorted_tickers,
    autopct="%1.1f%%",
    colors=sorted_colors,
    startangle=90,
    pctdistance=0.8,
    explode=[0.03] * len(sorted_rpct),
)
for text in autotexts:
    text.set_fontsize(10)
    text.set_fontweight("bold")
axes[0].set_title("Risk Contribution (%)\n(Component VaR / Portfolio VaR)",
                   fontsize=13)

# Right: Weight allocation pie for comparison
sorted_weights = weights[sorted_idx]
wedges2, texts2, autotexts2 = axes[1].pie(
    sorted_weights,
    labels=sorted_tickers,
    autopct="%1.1f%%",
    colors=sorted_colors,
    startangle=90,
    pctdistance=0.8,
    explode=[0.03] * len(sorted_weights),
)
for text in autotexts2:
    text.set_fontsize(10)
    text.set_fontweight("bold")
axes[1].set_title("Weight Allocation (%)", fontsize=13)

plt.suptitle("Risk Contribution vs Weight Allocation",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Compare the two charts: assets with high volatility or high")
print("correlation occupy a larger share of risk than their weight suggests.")

In [ ]:
# --- Grouped bar chart: weight % vs risk contribution % ---
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(tickers))
width = 0.35

weight_pct = weights * 100

bars1 = ax.bar(x - width/2, weight_pct, width,
               label="Weight %", color=PRIMARY, edgecolor="white", alpha=0.85)
bars2 = ax.bar(x + width/2, rpct, width,
               label="Risk Contribution %", color=SECONDARY, edgecolor="white", alpha=0.85)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9)

# Equal risk contribution reference line
erc_target = 100.0 / len(tickers)
ax.axhline(erc_target, color=TERTIARY, linestyle="--", linewidth=1.5,
           label=f"ERC target ({erc_target:.0f}%)")

ax.set_xlabel("Asset", fontsize=12)
ax.set_ylabel("Percentage (%)", fontsize=12)
ax.set_title("Weight Allocation vs Risk Contribution by Asset",
             fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(tickers, fontsize=11)
ax.legend(loc="upper right", frameon=True, framealpha=0.9)

plt.tight_layout()
plt.show()

print("The gap between weight % and risk contribution % reveals which")
print("assets are 'risk-efficient' (low risk relative to weight) vs")
print("'risk-intensive' (high risk relative to weight).")

---

## Conclusion: Risk Concentration & Diversification

This analysis reveals several important insights about portfolio risk:

### 1. Weight $\neq$ Risk Contribution

An asset's portfolio weight tells you how much capital is allocated, but **not** how much risk it contributes. A 10% position in a high-volatility, highly-correlated asset can contribute 25%+ of total portfolio risk. Conversely, a 30% position in a low-volatility diversifier might contribute only 15%.

### 2. Risk Concentration (HHI)

The Herfindahl-Hirschman Index of risk contributions measures how concentrated risk is:

$$
\text{HHI} = \sum_{i=1}^{n} \left(\frac{\text{CVaR}_i}{\text{VaR}}\right)^2
$$

- HHI $= 1/n$ means perfectly diversified (equal risk contribution).
- HHI $= 1$ means all risk is concentrated in a single asset.
- Most equity portfolios have HHI well above $1/n$ because volatility and correlation are uneven.

### 3. Marginal vs Component vs Incremental VaR

| Measure | Interpretation | Additive? |
|---------|---------------|-----------|
| **Marginal VaR** | Rate of change: $d\text{VaR}/dw_i$ | No (it's a derivative) |
| **Component VaR** | Additive contribution: $w_i \times \text{M-VaR}_i$ | Yes ($\sum \text{CVaR}_i = \text{VaR}$) |
| **Incremental VaR** | Impact of removing asset $i$ | No (removing changes diversification) |

### 4. Toward Risk Budgeting

The risk budget analysis shows how far the current portfolio deviates from an Equal Risk Contribution (ERC) target. Moving toward ERC typically means:
- **Reducing** weight in high-volatility, high-correlation assets.
- **Increasing** weight in low-volatility diversifiers.
- The resulting portfolio often looks very different from a market-cap or equal-weight benchmark, but provides more balanced risk exposure.

**Practical takeaway:** Always decompose portfolio risk into per-asset contributions before making allocation decisions. A portfolio that looks diversified by weight may be dangerously concentrated by risk.